In [5]:
# Part 3: CDR Mapping & Hotspot Identification
# AI-assisted in silico design of antibody variants targeting Influenza Hemagglutinin

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
import json

print("Part 3: CDR Mapping Analysis Started")
print("=" * 50)

Part 3: CDR Mapping Analysis Started


In [12]:
import os

print("Current directory:", os.getcwd())


file_path = "../part1_HA_dataset/5XKU.pdb"
print("File exists:", os.path.exists(file_path))


alternative_paths = [
    "5XKU.pdb",
    "../5XKU.pdb", 
    "../../5XKU.pdb",
    "/path/to/5XKU.pdb"
]

for path in alternative_paths:
    if os.path.exists(path):
        print(f"Found file at: {path}")
        break

Current directory: C:\Users\eceka\Influenza_1
File exists: False
Found file at: 5XKU.pdb


In [8]:
# Load 5XKU structure and extract Chain C sequence
parser = PDBParser()
structure = parser.get_structure("5XKU", "5XKU.pdb")  # Path düzeltildi

# Chain C sequence extraction  
chain_c = structure[0]['C']
sequence = []
residue_numbers = []

for residue in chain_c:
    if residue.get_id()[0] == ' ':  # Standard residues only
        sequence.append(seq1(residue.get_resname()))
        residue_numbers.append(residue.get_id()[1])

chain_c_sequence = ''.join(sequence)
print(f"Chain C Length: {len(chain_c_sequence)}")
print(f"Sequence: {chain_c_sequence}")
print(f"Residue range: {min(residue_numbers)} - {max(residue_numbers)}")

Chain C Length: 219
Sequence: QVQLQESGPGLVKPSETLSLTCTVSGGSISSGGYYWSWIRQHPGKGLEWIGYIYYSGSTDYNPSLKSRVTISVDTSKNQFSLKLSSVTAADTAVYYCAGGSTGDRHYYYYGMDVWGQGTTVTVSSASTKGPSVFPLAPSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLTQTYICNVNHKPSNTKVDKRVEP
Residue range: -3 - 221


C:\Users\eceka\anaconda3\Lib\site-packages\Bio\PDB\StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 5554.
  warnings.warn(
C:\Users\eceka\anaconda3\Lib\site-packages\Bio\PDB\StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 5769.
  warnings.warn(
C:\Users\eceka\anaconda3\Lib\site-packages\Bio\PDB\StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 5937.
  warnings.warn(


In [9]:
# CDR Prediction using Kabat numbering
# CDR regions for heavy chain (Chain C appears to be heavy chain based on length)

def predict_cdr_regions(sequence_length):
    """
    Predict CDR regions using Kabat numbering system
    Heavy chain CDR regions (approximate):
    - CDR1: 31-35
    - CDR2: 50-65  
    - CDR3: 95-102
    """
    
    # For 219 residue heavy chain
    cdr_regions = {
        'CDR1': (31, 35),
        'CDR2': (50, 65),
        'CDR3': (95, 102)
    }
    
    return cdr_regions

# Predict CDR regions
cdr_regions = predict_cdr_regions(len(chain_c_sequence))

print("Predicted CDR regions (Kabat numbering):")
for cdr, (start, end) in cdr_regions.items():
    length = end - start + 1
    print(f"{cdr}: {start}-{end} (Length: {length} residues)")
    
# Extract CDR sequences
for cdr, (start, end) in cdr_regions.items():
    # Adjust for 0-based indexing and negative numbering
    seq_start = max(0, start - min(residue_numbers))
    seq_end = min(len(chain_c_sequence), end - min(residue_numbers) + 1)
    
    if seq_start < len(chain_c_sequence) and seq_end > 0:
        cdr_seq = chain_c_sequence[seq_start:seq_end]
        print(f"{cdr} sequence: {cdr_seq}")

Predicted CDR regions (Kabat numbering):
CDR1: 31-35 (Length: 5 residues)
CDR2: 50-65 (Length: 16 residues)
CDR3: 95-102 (Length: 8 residues)
CDR1 sequence: YWSWI
CDR2 sequence: YYSGSTDYNPSLKSRV
CDR3 sequence: GGSTGDRH


In [10]:
# Interface analysis - which CDR residues are at the binding interface
# From Part 2: We know there are 34 interface residues

# Load interface data from Part 2 (if available) or calculate
print("Analyzing CDR-Interface overlap...")

# Simulate interface analysis (we'll do real PyMOL analysis next)
interface_residues = list(range(30, 40)) + list(range(55, 70)) + list(range(95, 105))

def analyze_cdr_interface_overlap(cdr_regions, interface_residues):
    """Analyze overlap between CDR regions and interface residues"""
    
    overlap_analysis = {}
    
    for cdr, (start, end) in cdr_regions.items():
        cdr_residues = set(range(start, end + 1))
        interface_set = set(interface_residues)
        
        overlap = cdr_residues.intersection(interface_set)
        overlap_count = len(overlap)
        total_cdr = len(cdr_residues)
        overlap_percent = (overlap_count / total_cdr) * 100 if total_cdr > 0 else 0
        
        overlap_analysis[cdr] = {
            'total_residues': total_cdr,
            'interface_residues': overlap_count,
            'overlap_percent': overlap_percent,
            'overlapping_positions': sorted(list(overlap))
        }
    
    return overlap_analysis

# Analyze overlap
overlap_results = analyze_cdr_interface_overlap(cdr_regions, interface_residues)

print("\nCDR-Interface Overlap Analysis:")
print("=" * 50)

for cdr, results in overlap_results.items():
    print(f"{cdr}:")
    print(f"  Total residues: {results['total_residues']}")
    print(f"  Interface residues: {results['interface_residues']}")
    print(f"  Overlap: {results['overlap_percent']:.1f}%")
    print(f"  Positions: {results['overlapping_positions']}")
    print()

Analyzing CDR-Interface overlap...

CDR-Interface Overlap Analysis:
CDR1:
  Total residues: 5
  Interface residues: 5
  Overlap: 100.0%
  Positions: [31, 32, 33, 34, 35]

CDR2:
  Total residues: 16
  Interface residues: 11
  Overlap: 68.8%
  Positions: [55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65]

CDR3:
  Total residues: 8
  Interface residues: 8
  Overlap: 100.0%
  Positions: [95, 96, 97, 98, 99, 100, 101, 102]



In [11]:
# Hotspot Prioritization Algorithm
def calculate_hotspot_scores(cdr_regions, overlap_results):
    """
    Calculate priority scores for mutation targets
    Scoring criteria:
    - CDR3: 3.0x weight (highest impact)
    - CDR1/CDR2: 2.0x weight  
    - Interface involvement: 2.5x bonus
    """
    
    hotspot_scores = {}
    
    # CDR importance weights
    cdr_weights = {'CDR1': 2.0, 'CDR2': 2.0, 'CDR3': 3.0}
    interface_bonus = 2.5
    
    for cdr, (start, end) in cdr_regions.items():
        cdr_weight = cdr_weights[cdr]
        overlap_percent = overlap_results[cdr]['overlap_percent']
        
        # Calculate base score
        base_score = cdr_weight
        
        # Add interface bonus
        interface_score = base_score * (overlap_percent / 100) * interface_bonus
        
        total_score = base_score + interface_score
        
        hotspot_scores[cdr] = {
            'base_score': base_score,
            'interface_bonus': interface_score,
            'total_score': total_score,
            'positions': list(range(start, end + 1))
        }
    
    return hotspot_scores

# Calculate scores
hotspot_scores = calculate_hotspot_scores(cdr_regions, overlap_results)

print("Hotspot Priority Scores:")
print("=" * 40)

# Sort by total score
sorted_cdrs = sorted(hotspot_scores.items(), key=lambda x: x[1]['total_score'], reverse=True)

for cdr, scores in sorted_cdrs:
    print(f"{cdr}: Score = {scores['total_score']:.2f}")
    print(f"  Base: {scores['base_score']:.1f} + Interface: {scores['interface_bonus']:.2f}")
    print(f"  Priority positions: {scores['positions']}")
    print()

# Identify top mutation targets
all_positions = []
for cdr, scores in sorted_cdrs:
    for pos in scores['positions']:
        all_positions.append((pos, cdr, scores['total_score']))

# Sort all positions by score
top_targets = sorted(all_positions, key=lambda x: x[2], reverse=True)[:10]

print("TOP 10 MUTATION TARGETS:")
print("=" * 30)
for i, (pos, cdr, score) in enumerate(top_targets, 1):
    print(f"{i:2d}. Position {pos} ({cdr}) - Score: {score:.2f}")

Hotspot Priority Scores:
CDR3: Score = 10.50
  Base: 3.0 + Interface: 7.50
  Priority positions: [95, 96, 97, 98, 99, 100, 101, 102]

CDR1: Score = 7.00
  Base: 2.0 + Interface: 5.00
  Priority positions: [31, 32, 33, 34, 35]

CDR2: Score = 5.44
  Base: 2.0 + Interface: 3.44
  Priority positions: [50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65]

TOP 10 MUTATION TARGETS:
 1. Position 95 (CDR3) - Score: 10.50
 2. Position 96 (CDR3) - Score: 10.50
 3. Position 97 (CDR3) - Score: 10.50
 4. Position 98 (CDR3) - Score: 10.50
 5. Position 99 (CDR3) - Score: 10.50
 6. Position 100 (CDR3) - Score: 10.50
 7. Position 101 (CDR3) - Score: 10.50
 8. Position 102 (CDR3) - Score: 10.50
 9. Position 31 (CDR1) - Score: 7.00
10. Position 32 (CDR1) - Score: 7.00
